In [2]:
import os

In [3]:
os.chdir('../')

In [6]:
%pwd

'/home/vk/Desktop/Python Code/Pytorch/End_to_end_Project/DS1/DataScienceProject_ETE'

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: str
    target_column: str
    mlflow_uri: str

In [9]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories, save_json


In [10]:
class ConfigurationManager:
    def __init__(self,
                config_filepath = CONFIG_FILE_PATH,
                params_filepath = PARAMS_FILE_PATH,
                schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evalution
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
    
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir, 
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_path, 
            target_column = schema.name, 
            mlflow_uri = "https://dagshub.com/kirshansharma3546/DataScienceProject_ETE.mlflow"



        )
        return model_evaluation_config

In [11]:
import os 
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import skops
import numpy as np
import joblib


In [ ]:
class ModelEvaluation:
    def __init__(self, config:ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2 

    def log_info_mlflow(self):
        
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            # Saving metrics as local
            scores = {"rmse": rmse, "mae":mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # Model registry does not work with file store 
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # pleae refer to the doc for more information 
                mlflow.sklearn.log_model(model, name = "model", registered_model_name="ElasticnetModel" , serialization_format="skops")

            else: 
                mlflow.sklearn.log_model(model, name = "model")


In [13]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_info_mlflow()
except Exception as e:
    raise e


[2026-04-30 23:30:26,464: INFO: common: yaml files: config/config.yaml loadded successfully]
[2026-04-30 23:30:26,467: INFO: common: yaml files: params.yaml loadded successfully]
[2026-04-30 23:30:26,469: INFO: common: yaml files: schema.yaml loadded successfully]
[2026-04-30 23:30:26,471: INFO: common: created directory at: artifacts]
[2026-04-30 23:30:26,472: INFO: common: created directory at: artifacts/model_evalution]
[2026-04-30 23:30:29,489: INFO: common: json file is saved at: artifacts/model_evalution/metrics.json]


2026/04/30 23:30:36 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpmd3bm0z8/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/04/30 23:30:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 5
Created version '5' of model 'ElasticnetModel'.


🏃 View run abrasive-hog-762 at: https://dagshub.com/kirshansharma3546/DataScienceProject_ETE.mlflow/#/experiments/0/runs/162533f391324b7cb71bbeb71ad68e73
🧪 View experiment at: https://dagshub.com/kirshansharma3546/DataScienceProject_ETE.mlflow/#/experiments/0
